# 04 — Fine-tuning DenseNet121 (3-fold OOF)

## 1. Clone từ GitHub

In [1]:
import subprocess
import sys
import importlib
from pathlib import Path

REPO_URL = "https://github.com/chienvoxn/The-Impostor-AI.git"
REPO_DIR = Path("/kaggle/working/The-Impostor-AI")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "master"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "master", REPO_URL, str(REPO_DIR)], check=True)

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import kemaodanh_oai.training as training
importlib.reload(training)
assert hasattr(training, "evaluate_pair_predictions"), "GitHub chưa có evaluate_pair_predictions; kiểm tra commit 73581e5"
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())
print("Pair evaluation import: OK")

Cloning into '/kaggle/working/The-Impostor-AI'...


Repository commit: 73581e5
Pair evaluation import: OK


## 2. Thiết lập GPU, seed và đọc dữ liệu development

In [2]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from kemaodanh_oai.dataset import load_development_data, make_image_dataframe, FaceClassificationDataset
from kemaodanh_oai.transforms import build_train_transform, build_valid_transform
from kemaodanh_oai.models import build_densenet121
from kemaodanh_oai.training import train_one_epoch, validate_one_epoch, evaluate_pair_predictions
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 20260917
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Hãy bật GPU T4 trong Kaggle Notebook Settings"
print("GPU:", torch.cuda.get_device_name(0))

pair_files = [p for p in Path("/kaggle/input").rglob("pairs.csv") if p.parent.name == "train"]
split_files = list(Path("/kaggle/input").rglob("development_split.csv"))
assert len(pair_files) == 1, f"Cần đúng 1 train/pairs.csv, thấy {len(pair_files)}"
assert len(split_files) == 1, f"Cần đúng 1 development_split.csv, thấy {len(split_files)}"
TRAIN_DIR = pair_files[0].parent
SPLIT_CSV = split_files[0]

dev_df = load_development_data(train_dir=TRAIN_DIR, split_csv=SPLIT_CSV)
image_df = make_image_dataframe(dev_df=dev_df, train_dir=TRAIN_DIR)
assert len(dev_df) == 800 and len(image_df) == 1600
assert set(dev_df["fold"].astype(int).unique()) == {0, 1, 2}
assert image_df["image_path"].map(lambda p: Path(p).is_file()).all(), "Thiếu ảnh train"
assert image_df.groupby("pair_id").size().eq(2).all()
print("Development pairs:", len(dev_df))
print("Development images:", len(image_df))
print("Pairs by fold:", dev_df["fold"].value_counts().sort_index().to_dict())
print("Holdout: không sử dụng trong notebook này")

GPU: Tesla T4
Development pairs: 800
Development images: 1600
Pairs by fold: {0: 267, 1: 267, 2: 266}
Holdout: không sử dụng trong notebook này


## 3. Cấu hình và thư mục đầu ra

Giữ `EPOCHS=20`, `PATIENCE=5` cho lần chạy chính thức. Mỗi fold dùng model mới từ ImageNet, không truyền trọng số fine-tuned từ fold trước.

In [3]:
from torch.utils.data import DataLoader
import gc
import time

EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 8
ACCUMULATION_STEPS = 3
NUM_WORKERS = 2
BACKBONE_LR = 2e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4

OUTPUT_DIR = Path("/kaggle/working/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_DIR = OUTPUT_DIR / "results"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Training plan: 3 folds, up to", EPOCHS, "epochs/fold")

Checkpoint directory: /kaggle/working/outputs/checkpoints
Results directory: /kaggle/working/outputs/results
Training plan: 3 folds, up to 20 epochs/fold


## 4. Huấn luyện đầy đủ 3 fold, lưu best + last sau mỗi epoch

**Đây là cell tốn thời gian GPU.** Nếu dừng giữa chừng, chạy lại từ đầu notebook; các fold đã hoàn tất sẽ được bỏ qua và fold đang dở tiếp tục từ `last` (nếu file còn). Không chạy song song hai phiên notebook trên cùng thư mục output.

In [4]:
def train_all_folds():
    for fold in (0, 1, 2):
        best_path = CHECKPOINT_DIR / f"densenet121_fold{fold}_best.pt"
        last_path = CHECKPOINT_DIR / f"densenet121_fold{fold}_last.pt"
        oof_path = RESULTS_DIR / f"densenet121_fold{fold}_oof.csv"
        done_path = RESULTS_DIR / f"densenet121_fold{fold}_done.txt"

        if done_path.exists() and best_path.exists() and oof_path.exists():
            print(f"Fold {fold}: đã hoàn thành, bỏ qua.")
            continue

        print(f"\n{'=' * 55}\nFINE-TUNING FOLD {fold}\n{'=' * 55}", flush=True)
        train_df = image_df[image_df["fold"] != fold].reset_index(drop=True)
        valid_df = image_df[image_df["fold"] == fold].reset_index(drop=True)
        assert set(train_df["pair_id"]).isdisjoint(set(valid_df["pair_id"]))
        assert len(valid_df) == (534 if fold in (0, 1) else 532)

        train_loader = DataLoader(
            FaceClassificationDataset(train_df, build_train_transform(224, 0.60)),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
        )
        valid_loader = DataLoader(
            FaceClassificationDataset(valid_df, build_valid_transform(224, 0.60)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
        )

        # Mỗi fold bắt đầu từ ImageNet, trừ khi resume checkpoint chính fold đó.
        model = build_densenet121(pretrained=True, num_classes=2).to(device)
        optimizer = torch.optim.AdamW([
            {"params": model.features.parameters(), "lr": BACKBONE_LR},
            {"params": model.classifier.parameters(), "lr": HEAD_LR},
        ], weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler("cuda", enabled=True)

        start_epoch = 1
        best_score, best_loss, best_epoch, bad_epochs = -1.0, float("inf"), 0, 0
        if last_path.exists():
            state = torch.load(last_path, map_location=device, weights_only=True)
            assert state["fold"] == fold
            model.load_state_dict(state["model_state_dict"])
            optimizer.load_state_dict(state["optimizer_state_dict"])
            scaler.load_state_dict(state["scaler_state_dict"])
            start_epoch = int(state["epoch"]) + 1
            best_score = float(state["best_macro_f1"])
            best_loss = float(state["best_valid_loss"])
            best_epoch = int(state["best_epoch"])
            bad_epochs = int(state["bad_epochs"])
            print(f"Resume fold {fold} từ epoch {start_epoch}")
            if not best_path.exists() or not oof_path.exists():
                raise FileNotFoundError("Thiếu best checkpoint hoặc OOF khi resume; hãy khôi phục output đồng bộ")

        for epoch in range(start_epoch, EPOCHS + 1):
            if bad_epochs >= PATIENCE:
                break
            started = time.time()
            train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler, accumulation_steps=ACCUMULATION_STEPS)
            valid_loss, valid_probs, valid_labels = validate_one_epoch(model, valid_loader, device)
            assert np.array_equal(valid_labels, valid_df["label"].to_numpy()), "Sai thứ tự validation"
            score, pair_results = evaluate_pair_predictions(valid_df, valid_probs, dev_df)
            assert len(pair_results) == len(valid_df) // 2
            print(f"Fold {fold} | Epoch {epoch:02d} | train={train_loss:.4f} | valid={valid_loss:.4f} | pair F1={score:.4f} | {(time.time()-started)/60:.1f} min", flush=True)

            improved = score > best_score or (score == best_score and valid_loss < best_loss)
            if improved:
                best_score, best_loss, best_epoch, bad_epochs = float(score), float(valid_loss), epoch, 0
                torch.save({
                    "fold": fold, "epoch": epoch, "model_state_dict": model.state_dict(),
                    "best_macro_f1": best_score, "best_valid_loss": best_loss,
                }, best_path)
                pair_results["fold"] = fold
                pair_results.to_csv(oof_path, index=False)
                print(f"  -> Đã lưu best checkpoint epoch {epoch}", flush=True)
            else:
                bad_epochs += 1

            torch.save({
                "fold": fold, "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "scaler_state_dict": scaler.state_dict(),
                "best_macro_f1": best_score, "best_valid_loss": best_loss,
                "best_epoch": best_epoch, "bad_epochs": bad_epochs,
            }, last_path)
            if bad_epochs >= PATIENCE:
                print(f"Early stopping fold {fold} ở epoch {epoch}", flush=True)
                break

        assert best_path.exists() and oof_path.exists(), f"Fold {fold} chưa có checkpoint tốt nhất"
        done_path.write_text(f"best_epoch={best_epoch}\nbest_macro_f1={best_score:.6f}\n", encoding="utf-8")
        print(f"FOLD {fold} XONG | best epoch={best_epoch} | best pair F1={best_score:.4f}", flush=True)
        del model, optimizer, scaler, train_loader, valid_loader
        gc.collect()
        torch.cuda.empty_cache()

train_all_folds()


FINE-TUNING FOLD 0
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 163MB/s] 


Training:   0%|          | 0/134 [00:00<?, ?it/s]

Validation:   0%|          | 0/67 [00:00<?, ?it/s]

Fold 0 | Epoch 01 | train=0.6285 | valid=0.5292 | pair F1=0.8501 | 2.0 min
  -> Đã lưu best checkpoint epoch 1


...

Validation:   0%|          | 0/67 [00:00<?, ?it/s]

Fold 2 | Epoch 19 | train=0.0344 | valid=0.2880 | pair F1=0.9474 | 0.2 min


Training:   0%|          | 0/134 [00:00<?, ?it/s]

Validation:   0%|          | 0/67 [00:00<?, ?it/s]

Fold 2 | Epoch 20 | train=0.0557 | valid=0.2915 | pair F1=0.9586 | 0.2 min
  -> Đã lưu best checkpoint epoch 20
FOLD 2 XONG | best epoch=20 | best pair F1=0.9586


## 5. Ghép OOF và đánh giá kết quả cuối cùng

In [5]:
oof_parts = []
for fold in (0, 1, 2):
    path = RESULTS_DIR / f"densenet121_fold{fold}_oof.csv"
    assert path.exists(), f"Chưa hoàn thành fold {fold}: {path}"
    assert (RESULTS_DIR / f"densenet121_fold{fold}_done.txt").exists(), f"Fold {fold} chưa hoàn thành"
    part = pd.read_csv(path, dtype={"pair_id": str})
    assert len(part) == (267 if fold in (0, 1) else 266)
    oof_parts.append(part)

oof_df = pd.concat(oof_parts, ignore_index=True)
assert len(oof_df) == 800 and oof_df["pair_id"].is_unique
assert set(oof_df["pair_id"]) == set(dev_df["pair_id"])

y_true = oof_df["true_fake_position"].astype(int)
y_pred = oof_df["pred_fake_position"].astype(int)
macro_f1 = f1_score(y_true, y_pred, average="macro")
accuracy = accuracy_score(y_true, y_pred)

full_oof_path = RESULTS_DIR / "densenet121_full_oof.csv"
oof_df.to_csv(full_oof_path, index=False)
print(f"FINAL 3-FOLD OOF MACRO-F1: {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"FINAL 3-FOLD OOF ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred, labels=[0, 1]))
print("\nClassification report:\n", classification_report(y_true, y_pred, labels=[0, 1], zero_division=0))
print("OOF saved:", full_oof_path)
display(oof_df.head())

FINAL 3-FOLD OOF MACRO-F1: 0.9650 (96.50%)
FINAL 3-FOLD OOF ACCURACY: 0.9650 (96.50%)

Confusion matrix:
 [[385  15]
 [ 13 387]]

Classification report:
               precision    recall  f1-score   support

           0       0.97      0.96      0.96       400
           1       0.96      0.97      0.97       400

    accuracy                           0.96       800
   macro avg       0.97      0.97      0.96       800
weighted avg       0.97      0.96      0.96       800

OOF saved: /kaggle/working/outputs/results/densenet121_full_oof.csv


,pair_id,prob_fake_0,prob_fake_1,true_fake_position,pred_fake_position,fold
0,00006,0.996901,0.082033,0,0,0
1,00009,0.321567,0.856392,1,1,0
2,00013,0.025299,0.998496,1,1,0
3,00021,0.174692,0.938067,1,1,0
4,00022,0.011464,0.943971,1,1,0


## 6. Đóng gói kết quả để tải xuống

In [6]:
import zipfile
zip_path = Path("/kaggle/working/densenet121_3fold_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for folder in (CHECKPOINT_DIR, RESULTS_DIR):
        for file in sorted(folder.rglob("*")):
            if file.is_file() and not file.name.endswith("_last.pt"):
                archive.write(file, arcname=str(file.relative_to(OUTPUT_DIR)))
print("Tải kết quả từ Kaggle Output:", zip_path)
print("Dung lượng ZIP (MB):", round(zip_path.stat().st_size / 1024**2, 1))
print("Lưu ý: ZIP chỉ chứa best checkpoints; các *_last.pt vẫn ở outputs/checkpoints để resume trong session.")

Tải kết quả từ Kaggle Output: /kaggle/working/densenet121_3fold_results.zip
Dung lượng ZIP (MB): 75.9
Lưu ý: ZIP chỉ chứa best checkpoints; các *_last.pt vẫn ở outputs/checkpoints để resume trong session.
